### 📚 Concept: Vector Databases and Similarity Search

#### 🔸 What is a Vector Database?
A **Vector Database** stores high-dimensional numerical vectors that represent data such as text, images, or audio. These vectors are often generated using embedding models (e.g., Cohere, OpenAI, Sentence Transformers) that convert raw data into dense numerical representations.

#### 🔸 Why Do We Use Vector Databases?
Vector DBs are designed for **efficient similarity search** across large collections of these vectors. They allow us to:
- Store and retrieve semantically similar documents
- Power search in recommendation systems
- Enable natural language search over structured or unstructured data
- Support retrieval-augmented generation (RAG) in LLM-based applications

---

#### 🔸 What is Similarity Search (a.k.a. Vector Search)?
Similarity search is the process of:
1. Converting a user query into a vector (embedding)
2. Comparing it with all stored vectors using a similarity metric (like cosine similarity or Euclidean distance)
3. Returning the most similar items/documents

---

#### 🔍 Similarity Metrics:
- **Cosine Similarity**: Measures angle between vectors (higher = more similar, range = [-1, 1])
- **Euclidean Distance (L2)**: Measures straight-line distance between vectors (lower = more similar)
- **Inner Product (Dot Product)**: Often used to approximate cosine similarity after normalization

---

#### ✅ Summary
Vector DBs combined with embedding models allow you to search and retrieve information **based on meaning**, not exact keywords. This enables powerful AI-driven applications such as:
- Semantic document search
- Question answering systems
- Chatbots with long-term memory
- Contextual autocomplete systems


## Required Imports

In [ ]:
# Import necessary libraries
from langchain_community.document_loaders import TextLoader
# Import necessary libraries
from langchain_community.vectorstores import FAISS
# Import necessary libraries
from langchain_cohere import CohereEmbeddings
# Import necessary libraries
from langchain_text_splitters import CharacterTextSplitter

### Load the raw text document


In [ ]:
# Load the text file using UTF-8 encoding
text_loader = TextLoader("speech.txt", encoding="utf-8")
raw_documents = text_loader.load()

### Split the document into smaller chunks for embedding

In [ ]:
# Split the document into manageable chunks
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
chunked_documents = splitter.split_documents(raw_documents)

### Initialize the Cohere Embeddings model

In [ ]:

COHERE_API_KEY = ""

# Initialize the Cohere embedding model
embedding_model = CohereEmbeddings(
    model="embed-english-v3.0",
    cohere_api_key=COHERE_API_KEY
)

### Create a FAISS vector store from the documents and embeddings

In [ ]:

vector_store = FAISS.from_documents(chunked_documents, embedding_model)

### Query definition

In [ ]:
# Define a natural language query
search_query = "How does the speaker describe the desired outcome of the war?"

### 1. Simple similarity search (returns list of relevant documents)

In [ ]:
# Perform a similarity search to find most relevant documents
similar_documents = vector_store.similarity_search(search_query)
# Display the result with content preview
print("\n🔍 Most Relevant Document Content:\n", similar_documents[0].page_content)

### 2. Using the retriever interface for querying

In [ ]:

retriever = vector_store.as_retriever()
retrieved_documents = retriever.invoke(search_query)

# Display the result with similarity score and content preview
print("\n📄 Retrieved Document:\n", retrieved_documents[0].page_content)

### 3. Similarity search with similarity scores

### Similarity Scores with FAISS (`IndexFlatL2`)

When using `FAISS.from_documents()` in LangChain, the default index type is `IndexFlatL2`, which performs **L2 (Euclidean) distance** calculations between embedding vectors.


#### Interpreting Scores

| **L2 Distance Score** | **Interpretation**                        |
|------------------------|-------------------------------------------|
| `~0.0`                 | Perfect match (very high similarity)      |
| `0.2 – 1.0`            | High similarity (strongly relevant)       |
| `1.0 – 2.0`            | Moderate similarity (partially relevant)  |
| `> 2.0`                | Low similarity (weak or unrelated)        |


In [ ]:

documents_with_scores = vector_store.similarity_search_with_score(search_query)

# Display the result with similarity score and content preview
print("\n📊 Similarity Search with Scores:")

for idx, (doc, score) in enumerate(documents_with_scores, start=1):
# Display the result with similarity score and content preview
    print(f"\n--- Result {idx} ---")
# Display the result with similarity score and content preview
    print(f"🔢 Similarity Score: {score:.4f}")
# Display the result with similarity score and content preview
    print(f"📄 Document Excerpt:\n{doc.page_content[:500]}...\n")

### 4. Convert the query to embedding and search using the vector

In [ ]:

# Define a natural language query
search_query = "How does the speaker describe the desired outcome of the war?"
# Convert the query into a vector embedding
query_vector = embedding_model.embed_query(search_query)

vector_based_results = vector_store.similarity_search_by_vector(query_vector)
# Display the result with similarity score and content preview
print("\n📈 Vector-Based Search Results:\n", vector_based_results)

### 5. Persist the FAISS index to disk (optional)

In [ ]:
vector_store.save_local("faiss_index")

### 6. Load the persisted index for future use

In [ ]:

loaded_vector_store = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

### 7. Perform a new search on the loaded index

In [ ]:

# Perform a similarity search to find most relevant documents
final_results = loaded_vector_store.similarity_search(search_query)
# Display the result with similarity score and content preview
print("\n📁 Final Search Results:\n", final_results[0].page_content)

## FAISS index for cosine similarity

### 📊 Understanding Cosine Similarity Scores

Cosine similarity measures the **angle** between two vectors in high-dimensional space. It's commonly used in semantic search and information retrieval to determine how similar a document is to a given query.

---

#### 🧠 Formula

\[
\text{cosine\_similarity} = \cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{||\vec{A}|| \cdot ||\vec{B}||}
\]

- Ranges from `-1.0` to `1.0`
  - `1.0`: vectors are identical → **perfect match**
  - `0.0`: vectors are orthogonal → **no similarity**
  - `-1.0`: vectors are opposite → **completely unrelated**

---

#### 📌 Interpreting Score: `0.4526`

This value indicates a **moderate level of semantic similarity**.

| **Cosine Score** | **Interpretation**                             |
|------------------|------------------------------------------------|
| `0.9 – 1.0`       | Very high similarity (near-perfect match)     |
| `0.7 – 0.9`       | High similarity (clearly related)             |
| `0.4 – 0.7`       | Moderate similarity (some overlap in meaning) |
| `0.2 – 0.4`       | Low similarity (minimally relevant)           |
| `< 0.2`           | Very low or no meaningful similarity          |



In [ ]:
# Import necessary libraries
from langchain_community.document_loaders import TextLoader
# Import necessary libraries
from langchain_community.vectorstores import FAISS
# Import necessary libraries
from langchain_cohere import CohereEmbeddings
# Import necessary libraries
from langchain_text_splitters import CharacterTextSplitter
# Import necessary libraries
import faiss
# Import necessary libraries
import numpy as np

# Step 1: Load the text document
# Load the text file using UTF-8 encoding
text_loader = TextLoader("speech.txt", encoding="utf-8")
raw_docs = text_loader.load()

# Step 2: Split the document into chunks
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
# Split the document into smaller chunks for embedding
documents = splitter.split_documents(raw_docs)

# Step 3: Initialize Cohere embedding model
# Initialize the Cohere embedding model
embedding_model = CohereEmbeddings(
    model="embed-english-v3.0",
    cohere_api_key=COHERE_API_KEY
)


# Step 4: Get document embeddings and normalize them
texts = [doc.page_content for doc in documents]
# Convert document text to embeddings
embeddings = np.array(embedding_model.embed_documents(texts)).astype("float32")
faiss.normalize_L2(embeddings)

# Step 5: Create FAISS index for cosine similarity
# Create a FAISS index using inner product (for cosine similarity)
index = faiss.IndexFlatIP(embeddings.shape[1])
# Add embeddings to the FAISS index
index.add(embeddings)


# Step 6: Convert query to normalized 2D embedding
# Define a natural language query
query = "How does the speaker describe the desired outcome of the war?"
# Convert the query into a vector embedding
query_vector = np.array(embedding_model.embed_query(query)).astype("float32").reshape(1, -1)
faiss.normalize_L2(query_vector)

# Step 7: Search for similar documents (top 3 results)
# Perform a similarity search to find most relevant documents
scores, indices = index.search(query_vector, k=3)

---

#### 📊 Interpretation of Each Score

- **0.4526** → This document has **moderate relevance**. It shares **key phrases or ideas**, but may not fully address the query.
- **0.4316** → Also moderately similar. Some **semantic alignment**, likely contains related context or supporting info.
- **0.4099** → On the **lower end of moderate similarity**. It may contain **only partial relevance** or tangential references.

---

In [ ]:
# Step 8: Show results
# Display the result with similarity score and content preview
print("\n📊 Cosine Similarity Search Results:")
for rank, idx in enumerate(indices[0], start=1):
    doc = documents[idx]
    score = scores[0][rank - 1]
# Display the result with similarity score and content preview
    print(f"\n--- Result {rank} ---")
# Display the result with similarity score and content preview
    print(f"🧭 Cosine Similarity Score: {score:.4f}")
# Display the result with similarity score and content preview
    print(f"📄 Document Preview:\n{doc.page_content[:500]}...\n")